# Mock 实践

用 pytest 测试用例学习返回值、异常和调用验证，不连接设备。每个用例独立创建替身，异常用 `pytest.raises` 验证。

替身的选择见 [[learning/testing/methods#如何选择|Stub、Mock 与 Fake]]。`return_value`、`side_effect` 是属性，`assert_*` 是断言方法。


## 准备

Mock 动态创建子 Mock，调用时记录参数。`Device` 只描述接口，不实现设备读写。


In [ ]:
import pytest

from typing import Protocol
from unittest.mock import AsyncMock, DEFAULT, MagicMock, Mock, call, create_autospec, patch


class Device(Protocol):
    def write_config(self, value: bytes) -> None: ...
    def read_config(self) -> bytes: ...


## 构造与行为配置

`return_value`、`side_effect` 既可以在构造时传入，也可以在创建后赋值；`spec`、`spec_set` 等接口约束在构造时指定。每个参数只在下面对应的小节介绍。


### return_value

设置调用返回值，预设时不会触发调用。`Mock(return_value=...)` 控制 `mock()`，子方法的返回值要单独配置。固定返回值不会与写入内容自动关联。


In [ ]:
def test_return_value() -> None:
    read = Mock(return_value=b"\x01\x02")
    read.assert_not_called()
    assert read() == b"\x01\x02"

    device = Mock(spec=Device)
    device.read_config.return_value = b"\x01\x02"
    device.write_config(b"\xff\xff")
    assert device.read_config() == b"\x01\x02"


### side_effect：异常

赋值时不抛异常；调用写入方法时才抛出。Mock 仍会记录这次失败的调用。


In [ ]:
def test_side_effect_exception() -> None:
    device = Mock(spec=Device)
    device.write_config.side_effect = TimeoutError('设备超时')
    with pytest.raises(TimeoutError):
        device.write_config(b'\x01\x02')
    device.write_config.assert_called_once_with(b'\x01\x02')


### side_effect：列表

每次调用消耗一个元素，异常元素会被抛出；耗尽后抛出 `StopIteration`。这里是手动调用三次，Mock 不会自动重试。


In [ ]:
def test_side_effect_sequence() -> None:
    device = Mock(spec=Device)
    device.read_config.side_effect = [TimeoutError('首次读取超时'), b'\x01\x02']
    with pytest.raises(TimeoutError):
        device.read_config()
    assert device.read_config() == b'\x01\x02'
    with pytest.raises(StopIteration):
        device.read_config()


### side_effect：函数

赋值使用函数名，不加括号。调用 Mock 时，实际参数会传给这个函数，由它决定返回值或异常。


In [ ]:
def test_side_effect_function() -> None:
    def check_write(value: bytes) -> None:
        if len(value) != 2:
            raise ValueError('配置字必须是两个字节')
          
    device = Mock(spec=Device)
    device.write_config.side_effect = check_write
    assert device.write_config(b'\x01\x02') is None
    with pytest.raises(ValueError):
        device.write_config(b'\x01')


### side_effect 与 return_value

`side_effect` 优先；函数返回 `DEFAULT` 时才继续使用 `return_value`。返回 `None` 就是返回 `None`，不会回退。

`side_effect = None` 关闭副作用，继续使用 `return_value`；它不同于 `side_effect = lambda: None`。


In [ ]:
def test_side_effect_precedence() -> None:
    read = Mock(return_value=b'\x01\x02')
    read.side_effect = lambda : b'\xff\xff'
    assert read() == b'\xff\xff'
    read.side_effect = lambda : DEFAULT
    assert read() == b'\x01\x02'
    read.side_effect = None
    assert read() == b'\x01\x02'


### spec 与 spec_set

`spec=Device` 限制可访问的属性；`spec_set=Device` 还禁止设置未知属性。这里给 `spec_set` 传接口，不是 `True`。

两者都不检查方法参数签名；需要检查签名时用 `create_autospec`，它不是 `Mock` 的构造参数。


In [ ]:
def test_constructor_constraints() -> None:
    device = Mock(spec=Device)
    with pytest.raises(AttributeError):
        device.missing_method()
    device.extra = True
    assert device.extra is True

    strict_device = Mock(spec_set=Device)
    with pytest.raises(AttributeError):
        strict_device.extra = True


### create_autospec

进一步检查方法签名，例如缺少必填参数。类型标注不会因此变成运行时类型校验。


In [ ]:
def test_autospec() -> None:
    device = create_autospec(Device, instance=True, spec_set=True)
    device.write_config(b'\x01\x02')
    with pytest.raises(TypeError):
        device.write_config()
    device.write_config('字符串也符合参数数量，类型标注不会自动检查')


### name

调试名称会显示在对象表示和部分断言错误中，也会传递给子 Mock。它不是名为 `name` 的业务属性。


In [ ]:
def test_constructor_name() -> None:
    device = Mock(spec=Device, name="primary_device")
    assert "primary_device" in repr(device)
    assert "primary_device.read_config" in repr(device.read_config)


### wraps

包装真实函数或对象，在记录调用的同时执行真实逻辑。不要随意包装真实设备，否则仍会操作设备。

显式设置 `return_value` 后，不再调用被包装函数。通常优先级是 `side_effect` → `return_value` → `wraps`。


In [ ]:
def test_constructor_wraps() -> None:
    values = []
    append = Mock(wraps=values.append)
    assert append(b"\x01\x02") is None
    assert values == [b"\x01\x02"]
    append.assert_called_once_with(b"\x01\x02")

    append.return_value = None
    append(b"\x03\x04")
    assert values == [b"\x01\x02"]
    assert append.call_count == 2


### kwargs

额外关键字参数可以设置普通属性，用 `**{...}` 还能配置子方法。配置较多时，分行赋值往往更清楚。


In [ ]:
def test_constructor_attributes() -> None:
    device = Mock(
        host="192.0.2.1",
        **{
            "read_config.return_value": b"\x01\x02",
            "write_config.side_effect": TimeoutError("写入超时"),
        },
    )
    assert device.host == "192.0.2.1"
    assert device.read_config() == b"\x01\x02"
    with pytest.raises(TimeoutError, match="写入超时"):
        device.write_config(b"\x01\x02")


### unsafe

默认 `unsafe=False`，阻止部分以 `assert` 或其常见拼写错误开头的未知属性，防止写错断言却悄悄通过。

`unsafe=True` 允许动态创建这些属性，通常不建议开启；它不会解除 `spec` / `spec_set` 的限制。


In [ ]:
def test_constructor_unsafe() -> None:
    device = Mock()
    with pytest.raises(AttributeError):
        device.assert_ready()

    permissive_device = Mock(unsafe=True)
    permissive_device.assert_ready()
    permissive_device.assert_ready.assert_called_once_with()


## assert_called_once_with

检查已经发生的调用是否恰好一次且参数匹配；断言本身不会再调用设备方法。


In [ ]:
def test_assert_called_once_with() -> None:
    device = Mock(spec=Device)
    device.write_config(b'\x01\x02')
    device.write_config.assert_called_once_with(b'\x01\x02')


## assert_called_with

只检查最后一次调用参数，不限制调用次数。


In [ ]:
def test_assert_called_with() -> None:
    device = Mock(spec=Device)
    device.write_config(b'\x01\x02')
    device.write_config(b'\x03\x04')
    device.write_config.assert_called_with(b'\x03\x04')
    assert device.write_config.call_count == 2


## assert_any_call

检查某次调用是否使用过指定参数，不要求它是最后一次。


In [ ]:
def test_assert_any_call() -> None:
    device = Mock(spec=Device)
    device.write_config(b'\x01\x02')
    device.write_config(b'\x03\x04')
    device.write_config.assert_any_call(b'\x01\x02')


## assert_not_called

检查方法从未被调用，可用于确认输入校验失败或写入失败后没有继续读设备。


In [ ]:
def test_assert_not_called() -> None:
    device = Mock(spec=Device)
    device.write_config(b'\x01\x02')
    device.read_config.assert_not_called()


## called 与 call_count

分别查看是否调用过、调用多少次。`assert_called()` 检查至少一次，`assert_called_once()` 检查恰好一次，都不检查参数。


In [ ]:
def test_call_count() -> None:
    read = Mock(return_value=b'\x01\x02')
    assert not read.called
    assert read.call_count == 0
    read()
    read.assert_called()
    read.assert_called_once()
    assert read.called
    assert read.call_count == 1


## call_args 与 call_args_list

读取最后一次或全部调用记录。`.args` 是位置参数，`.kwargs` 是关键字参数。


In [ ]:
def test_call_args() -> None:
    device = Mock(spec=Device)
    device.write_config(b'\x01\x02')
    device.write_config(value=b'\x03\x04')
    assert device.write_config.call_args.args == ()
    assert device.write_config.call_args.kwargs == {'value': b'\x03\x04'}
    assert device.write_config.call_args_list == [call(b'\x01\x02'), call(value=b'\x03\x04')]


## mock_calls

查看父 Mock 和子 Mock 的调用。精确比较列表可同时验证跨方法顺序，并拒绝额外调用。


In [ ]:
def test_mock_calls() -> None:
    device = Mock(spec=Device)
    device.read_config.return_value = b'\x01\x02'
    device.write_config(b'\x01\x02')
    device.read_config()
    assert device.mock_calls == [call.write_config(b'\x01\x02'), call.read_config()]


## assert_has_calls

默认匹配连续且有序的一段调用，允许前后有其他调用。`any_order=True` 不要求顺序；如需完整精确匹配，用 `mock_calls == [...]`。


In [ ]:
def test_assert_has_calls() -> None:
    device = Mock(spec=Device)
    device.write_config(b'\x00\x01')
    device.write_config(b'\x01\x02')
    device.read_config()
    device.write_config(b'\x03\x04')
    device.assert_has_calls([call.write_config(b'\x01\x02'), call.read_config()])
    device.assert_has_calls([call.read_config(), call.write_config(b'\x01\x02')], any_order=True)


## reset_mock

默认清除调用记录，保留行为配置；传入 `return_value=True, side_effect=True` 才重置它们。每个用例新建替身通常更容易理解。


In [ ]:
def test_reset_mock() -> None:
    read = Mock(return_value=b'\x01\x02', side_effect=lambda : DEFAULT)
    read()
    read.reset_mock()
    read.assert_not_called()
    assert read() == b'\x01\x02'
    read.reset_mock(return_value=True, side_effect=True)
    read.assert_not_called()
    assert read.side_effect is None
    assert isinstance(read(), Mock)


## AsyncMock

模拟异步方法。被调用不等于已被等待，使用 `assert_awaited_once_with()` 检查 `await`。`@pytest.mark.asyncio` 让 pytest-asyncio 执行异步测试函数。

Mock 对象可以像函数一样调用，不一定要写成 `对象.方法()`。

- `read_async = AsyncMock(return_value=...)` 是创建替身、配置返回值，不计入调用记录。
- `pending = read_async(timeout=1.0)` 才是调用替身本身，因此可以用 `assert_called_once_with(timeout=1.0)` 检查调用参数。
- `await pending` 后，再用 `assert_awaited_once_with(timeout=1.0)` 检查等待记录。


In [ ]:
@pytest.mark.asyncio
async def test_async_mock() -> None:
    read_async = AsyncMock(return_value=b'\x01\x02')
    pending = read_async(timeout=1.0)
    read_async.assert_called_once_with(timeout=1.0)
    read_async.assert_not_awaited()
    result = await pending
    assert result == b'\x01\x02'
    read_async.assert_awaited_once_with(timeout=1.0)


## MagicMock

需要 `with`、迭代等特殊方法时使用。这里进入上下文返回设备，退出和迭代都不访问真实资源。


In [ ]:
def test_magic_mock() -> None:
    session = MagicMock()
    session.__enter__.return_value = Mock(spec=Device)
    with session as device:
        device.write_config(b'\x01\x02')
    session.__enter__.assert_called_once_with()
    session.__exit__.assert_called_once_with(None, None, None)
    device.write_config.assert_called_once_with(b'\x01\x02')
    items = MagicMock()
    items.__iter__.return_value = iter(['primary', 'secondary'])
    assert list(items) == ['primary', 'secondary']


## patch

替换被测代码查找依赖的位置，并在退出 `with` 后恢复。这里函数使用 `time.sleep`，因此替换该属性。若代码使用 `from time import sleep`，应替换使用方模块的 `sleep`。


In [ ]:
def test_patch() -> None:
    import time

    def wait_before_read() -> None:
        time.sleep(10)
    original_sleep = time.sleep
    with patch('time.sleep') as sleep_mock:
        wait_before_read()
        sleep_mock.assert_called_once_with(10)
    assert time.sleep is original_sleep


## 使用边界

- Mock 的断言证明代码按约定调用接口，不证明配置真实写入了设备。
- 只有次数、顺序属于业务约定时才检查，避免测试绑定内部实现。
- 需要写入与回读关联时，优先使用简单 Fake。
- 已支持构造注入的服务直接传替身，不必再用 `patch`。
